# Análise de unidades compradoras

Baixa o CSV de itens hospedado no Google Drive (acesso restrito, exige login)
para `tasks/analise-unidades-compradoras/tmp/` e carrega os dados para análise.

**Autenticação:** defina `GOOGLE_DRIVE_CREDENTIALS` com o caminho de um JSON de credenciais
(padrão: `google-drive-credentials.json` na raiz do repositório). São aceitos:

- *OAuth client (Desktop app)*: abre o navegador para login na primeira execução e guarda o token em `tmp/token-drive.json`;
- *Service account*: o arquivo do Drive precisa estar compartilhado com o e-mail da conta de serviço.

Dependências: `pip install -r requirements.txt`.

In [1]:
import io
import json
import os
from pathlib import Path

import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

## Configuração

In [2]:
def encontra_raiz_repositorio(inicio: Path) -> Path:
    for pasta in [inicio, *inicio.parents]:
        if (pasta / ".git").exists():
            return pasta
    raise FileNotFoundError("Raiz do repositório não encontrada a partir de " + str(inicio))


REPO_ROOT = encontra_raiz_repositorio(Path.cwd().resolve())
TASK_DIR = REPO_ROOT / "tasks" / "analise-unidades-compradoras"
TMP_DIR = TASK_DIR / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FILE_ID = "1Ang6CC87uEBFZFFbVKdxZry1iVphzs1q"
ARQUIVO_ITENS = TMP_DIR / "itens.csv"

CREDENCIAIS_PATH = Path(
    os.environ.get("GOOGLE_DRIVE_CREDENTIALS", REPO_ROOT / "google-drive-credentials.json")
)
TOKEN_PATH = TMP_DIR / "token-drive.json"
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

## Login no Google Drive

In [ ]:
def autentica_drive():
    if not CREDENCIAIS_PATH.exists():
        raise FileNotFoundError(
            f"Credenciais não encontradas em {CREDENCIAIS_PATH}. "
            "Defina GOOGLE_DRIVE_CREDENTIALS ou coloque o JSON nesse caminho."
        )

    with open(CREDENCIAIS_PATH, encoding="utf-8") as arquivo:
        tipo = json.load(arquivo).get("type")

    if tipo == "service_account":
        credenciais = service_account.Credentials.from_service_account_file(
            CREDENCIAIS_PATH, scopes=SCOPES
        )
    else:
        credenciais = None
        if TOKEN_PATH.exists():
            credenciais = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)
        if credenciais and credenciais.expired and credenciais.refresh_token:
            credenciais.refresh(Request())
        if not credenciais or not credenciais.valid:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENCIAIS_PATH, SCOPES)
            credenciais = flow.run_local_server(port=0)
        TOKEN_PATH.write_text(credenciais.to_json(), encoding="utf-8")

    return build("drive", "v3", credentials=credenciais)


drive = autentica_drive()

## Download do CSV de itens

In [ ]:
def baixa_arquivo_drive(drive, file_id: str, destino: Path) -> Path:
    metadados = drive.files().get(
        fileId=file_id, fields="name, size, mimeType", supportsAllDrives=True
    ).execute()
    print(f"Arquivo no Drive: {metadados['name']} ({metadados.get('mimeType')})")

    arquivo_parcial = destino.with_suffix(destino.suffix + ".part")
    requisicao = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(arquivo_parcial, "wb") as saida:
        downloader = MediaIoBaseDownload(saida, requisicao, chunksize=50 * 1024 * 1024)
        concluido = False
        while not concluido:
            status, concluido = downloader.next_chunk()
            if status:
                print(f"  {status.progress():.0%}")

    arquivo_parcial.replace(destino)
    print(f"Salvo em {destino} ({destino.stat().st_size / 1024 / 1024:.1f} MB)")
    return destino


baixa_arquivo_drive(drive, DRIVE_FILE_ID, ARQUIVO_ITENS)

## Leitura dos itens

In [3]:
itens = pd.read_csv('tmp/itens.csv', low_memory=False)
print(itens.shape)
itens.dtypes

(154402, 52)


numeroItem                                    int64
endpoint                                        str
descricao                                       str
materialOuServico                               str
materialOuServicoNome                           str
valorUnitarioEstimado                       float64
valorTotal                                  float64
quantidade                                  float64
unidadeMedida                                   str
orcamentoSigiloso                              bool
itemCategoriaId                               int64
itemCategoriaNome                               str
patrimonio                                      str
codigoRegistroImobiliario                       str
criterioJulgamentoId                          int64
criterioJulgamentoNome                          str
situacaoCompraItem                            int64
situacaoCompraItemNome                          str
tipoBeneficio                               float64
tipoBenefici

In [4]:
itens.head()

,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.dataAtualizacao,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia
0,1,https://pncp.gov.br/api/pncp/v1/orgaos/0157259...,ONIBUS MARCOLPOLO VOLARE placa CDV 2676,M,Material,35000.0,35000.0,1.0,UN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
1,2,https://pncp.gov.br/api/pncp/v1/orgaos/0157259...,PEUGEOT PARTNER 1.6 placa GKC 1574,M,Material,15000.0,15000.0,1.0,UN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
2,1,https://pncp.gov.br/api/pncp/v1/orgaos/8984894...,ALIENAÇÃO DE BEM IMÓVEL - LOTE URBANO Nº 07 DA...,M,Material,2920091.2,2920091.2,1.0,UNIDADE (UN),False,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
3,1,https://pncp.gov.br/api/pncp/v1/orgaos/5415852...,"CAMINHÃO MARCA VOLKSWAGEN, MODELO 19-360 CONST...",M,Material,120000.0,120000.0,1.0,UN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
4,2,https://pncp.gov.br/api/pncp/v1/orgaos/5415852...,UNIDADE MÓVEL DE BRITAGEM (MODELO BIM 100900 -...,M,Material,100000.0,100000.0,1.0,UN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN


## Itens por CNPJ do órgão

O CSV de itens não tem coluna própria de CNPJ: ele é extraído da coluna `endpoint`
(`.../orgaos/<CNPJ>/compras/<ano>/<sequencial>/itens`).

In [5]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "25.089.137/0001-95"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade01 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade01)} itens em {itens_cnpj_unidade01['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade01

2 itens em 1 compra(s) do CNPJ 25.089.137/0001-95


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao
42594,1,https://pncp.gov.br/api/pncp/v1/orgaos/2508913...,Treinamento Qualificação Profissional,S,Serviço,0.0,0.0,1.0,UNIDADE,True,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,25089137000195
42595,2,https://pncp.gov.br/api/pncp/v1/orgaos/2508913...,Treinamento Qualificação Profissional,S,Serviço,0.0,0.0,1.0,UNIDADE,True,...,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,25089137000195


In [6]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "16.723.250/0001-9"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade02 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade02)} itens em {itens_cnpj_unidade02['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade02

0 itens em 0 compra(s) do CNPJ 16.723.250/0001-9


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao


In [7]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "18.729.020/0001-63"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade03 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade03)} itens em {itens_cnpj_unidade03['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade03

0 itens em 0 compra(s) do CNPJ 18.729.020/0001-63


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao


In [8]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "18.729.020/0002-44"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade03_1 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade03_1)} itens em {itens_cnpj_unidade03_1['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade03_1

0 itens em 0 compra(s) do CNPJ 18.729.020/0002-44


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao


In [9]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "18.729.020/0004-06"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade03_2 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade03_2)} itens em {itens_cnpj_unidade03_2['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade03_2

0 itens em 0 compra(s) do CNPJ 18.729.020/0004-06


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao


In [10]:
itens["cnpj_orgao"] = itens["endpoint"].str.extract(r"/orgaos/(\d{14})/", expand=False)

CNPJ = "18.729.020/0005-97"
cnpj_limpo = "".join(filter(str.isdigit, CNPJ))

itens_cnpj_unidade03_3 = itens[itens["cnpj_orgao"] == cnpj_limpo].copy()
print(f"{len(itens_cnpj_unidade03_3)} itens em {itens_cnpj_unidade03_3['endpoint'].nunique()} compra(s) do CNPJ {CNPJ}")
itens_cnpj_unidade03_3

0 itens em 0 compra(s) do CNPJ 18.729.020/0005-97


,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,categoriaItemCatalogo.statusAtivo,catalogoCodigoItem,informacaoComplementar,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,exigenciaConteudoNacional,catalogo,categoriaItemCatalogo,tipoMargemPreferencia,cnpj_orgao


## Leitura das contratações

`tmp/contratacoes.csv` vem do endpoint `contratacoes/atualizacao` do PNCP, então traz as contratações
**atualizadas** na janela coletada (veja `endpoint`), inclusive publicações antigas que foram atualizadas nesse período.

Colunas de identificação do órgão:

- `data.orgaoEntidade.cnpj` / `data.orgaoEntidade.razaoSocial`: órgão responsável pela contratação;
- `data.unidadeOrgao.codigoUnidade` / `data.unidadeOrgao.nomeUnidade`: unidade compradora;
- `data.orgaoSubRogado.cnpj`: órgão sub-rogado, quando houver.

In [13]:
ARQUIVO_CONTRATACOES = TMP_DIR / "contratacoes.csv"

contratacoes = pd.read_csv(ARQUIVO_CONTRATACOES, dtype=str)
print(contratacoes.shape)

(49909, 68)


## Contratações por CNPJ

Análise feita apenas com o CSV de contratações. Para cada CNPJ da lista `CNPJS`, reúne as contratações em que
ele é o órgão responsável (`data.orgaoEntidade.cnpj`) ou o órgão sub-rogado (`data.orgaoSubRogado.cnpj`),
as unidades compradoras envolvidas e outros CNPJs da mesma raiz (8 primeiros dígitos, ou seja, filiais do mesmo órgão).

Para analisar outros órgãos, edite `CNPJS` (com ou sem pontuação) e rode a seção novamente.

In [20]:
CNPJS = {
    "unidade01": "25089137000195",
    "unidade02": "16723250000190",
    "unidade03": "18729020000163",
    "unidade03_1": "18729020000244",
    "unidade03_2": "18729020000406",
    "unidade03_3": "18729020000597",
}

COLUNAS_UNIDADE = [
    "data.unidadeOrgao.codigoUnidade",
    "data.unidadeOrgao.nomeUnidade",
    "data.unidadeOrgao.municipioNome",
    "data.unidadeOrgao.ufSigla",
]


def limpa_cnpj(cnpj: str) -> str:
    return "".join(filter(str.isdigit, cnpj))


def cnpj_valido(cnpj: str) -> bool:
    if len(cnpj) != 14 or len(set(cnpj)) == 1:
        return False
    for tamanho, pesos in ((12, [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]), (13, [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])):
        resto = sum(int(d) * p for d, p in zip(cnpj[:tamanho], pesos)) % 11
        if int(cnpj[tamanho]) != (0 if resto < 2 else 11 - resto):
            return False
    return True


def analisa_contratacoes_cnpj(cnpj: str) -> dict:
    cnpj_limpo = limpa_cnpj(cnpj)

    eh_orgao = contratacoes["data.orgaoEntidade.cnpj"] == cnpj_limpo
    eh_subrogado = contratacoes["data.orgaoSubRogado.cnpj"] == cnpj_limpo
    contratacoes_cnpj = contratacoes[eh_orgao | eh_subrogado].copy()
    contratacoes_cnpj.insert(
        0, "papel_cnpj", eh_orgao[eh_orgao | eh_subrogado].map({True: "órgão responsável", False: "órgão sub-rogado"})
    )

    unidades_cnpj = (
        contratacoes_cnpj.groupby(COLUNAS_UNIDADE, dropna=False)
        .agg(
            contratacoes=("data.numeroControlePNCP", "nunique"),
            valor_total_estimado=("data.valorTotalEstimado", lambda v: pd.to_numeric(v, errors="coerce").sum()),
        )
        .reset_index()
    )

    cnpjs_contratacoes = pd.concat(
        [contratacoes["data.orgaoEntidade.cnpj"], contratacoes["data.orgaoSubRogado.cnpj"]]
    ).dropna().unique()
    mesma_raiz = sorted(c for c in cnpjs_contratacoes if c[:8] == cnpj_limpo[:8] and c != cnpj_limpo)

    return {
        "cnpj": cnpj_limpo,
        "cnpj_valido": cnpj_valido(cnpj_limpo),
        "contratacoes": contratacoes_cnpj,
        "unidades": unidades_cnpj,
        "mesma_raiz": mesma_raiz,
    }

### Resumo por CNPJ

In [21]:
resultados_contratacoes = {nome: analisa_contratacoes_cnpj(cnpj) for nome, cnpj in CNPJS.items()}

resumo_contratacoes_cnpjs = pd.DataFrame(
    [
        {
            "nome": nome,
            "cnpj_informado": CNPJS[nome],
            "cnpj_valido": r["cnpj_valido"],
            "razao_social": r["contratacoes"]["data.orgaoEntidade.razaoSocial"].iloc[0] if len(r["contratacoes"]) else None,
            "contratacoes": len(r["contratacoes"]),
            "unidades_compradoras": len(r["unidades"]),
            "outros_cnpjs_mesma_raiz": ", ".join(r["mesma_raiz"]),
        }
        for nome, r in resultados_contratacoes.items()
    ]
)
resumo_contratacoes_cnpjs

,nome,cnpj_informado,cnpj_valido,razao_social,contratacoes,unidades_compradoras,outros_cnpjs_mesma_raiz
0,unidade01,25089137000195,True,"SECRETARIA DA AGRICULTURA, PECUARIA E AQUICULTURA",1,1,
1,unidade02,16723250000190,True,NaN,0,0,
2,unidade03,18729020000163,True,NaN,0,0,
3,unidade03_1,18729020000244,True,NaN,0,0,
4,unidade03_2,18729020000406,True,NaN,0,0,
5,unidade03_3,18729020000597,True,NaN,0,0,


### Contratações e unidades de todos os CNPJs

A coluna `nome` indica de qual entrada de `CNPJS` cada linha veio. Para ver um CNPJ isolado,
use por exemplo `resultados_contratacoes["unidade01"]["contratacoes"]`.

In [16]:
def consolida(chave: str) -> pd.DataFrame:
    df = pd.concat([r[chave].assign(nome=nome) for nome, r in resultados_contratacoes.items()], ignore_index=True)
    return df[["nome", *df.columns.drop("nome")]]


contratacoes_cnpjs = consolida("contratacoes")
unidades_cnpjs = consolida("unidades")

In [17]:
contratacoes_cnpjs[
    [
        "nome",
        "papel_cnpj",
        "data.numeroControlePNCP",
        "data.orgaoEntidade.razaoSocial",
        *COLUNAS_UNIDADE[:3],
        "data.modalidadeNome",
        "data.situacaoCompraNome",
        "data.dataPublicacaoPncp",
        "data.valorTotalEstimado",
        "data.valorTotalHomologado",
        "data.objetoCompra",
    ]
]

,nome,papel_cnpj,data.numeroControlePNCP,data.orgaoEntidade.razaoSocial,data.unidadeOrgao.codigoUnidade,data.unidadeOrgao.nomeUnidade,data.unidadeOrgao.municipioNome,data.modalidadeNome,data.situacaoCompraNome,data.dataPublicacaoPncp,data.valorTotalEstimado,data.valorTotalHomologado,data.objetoCompra
0,unidade01,órgão responsável,25089137000195-1-000010/2026,"SECRETARIA DA AGRICULTURA, PECUARIA E AQUICULTURA",925962,"ETO-SECRET. AGRICULTURA, PESCA E ABASTECIMEN.",Palmas,Pregão - Eletrônico,Divulgada no PNCP,2026-02-23T07:03:30,0,NaN,Contratação de empresa ou instituição especial...


In [18]:
unidades_cnpjs

,nome,data.unidadeOrgao.codigoUnidade,data.unidadeOrgao.nomeUnidade,data.unidadeOrgao.municipioNome,data.unidadeOrgao.ufSigla,contratacoes,valor_total_estimado
0,unidade01,925962,"ETO-SECRET. AGRICULTURA, PESCA E ABASTECIMEN.",Palmas,TO,1,0


### Contratações filtradas pelos CNPJs

CSV de contratações com apenas as linhas cujo CNPJ do órgão (`data.orgaoEntidade.cnpj`) está em `CNPJS`.

In [19]:
cnpjs_limpos = [limpa_cnpj(cnpj) for cnpj in CNPJS.values()]

contratacoes_filtradas = contratacoes[contratacoes["data.orgaoEntidade.cnpj"].isin(cnpjs_limpos)].copy()
print(f"{len(contratacoes_filtradas)} de {len(contratacoes)} contratações batem com os CNPJs informados")
contratacoes_filtradas

1 de 49909 contratações batem com os CNPJs informados


,data.srp,data.orgaoEntidade,data.orgaoEntidade.cnpj,data.orgaoEntidade.razaoSocial,data.orgaoEntidade.poderId,data.orgaoEntidade.esferaId,data.anoCompra,data.sequencialCompra,data.dataInclusao,data.dataPublicacaoPncp,...,data.situacaoCompraId,data.situacaoCompraNome,data.usuarioNome,data.emendaParlamentar,totalRegistros,totalPaginas,numeroPagina,paginasRestantes,empty,endpoint
9059,FALSE,NaN,25089137000195,"SECRETARIA DA AGRICULTURA, PECUARIA E AQUICULTURA",E,E,2026,10,2026-02-23T07:03:30,2026-02-23T07:03:30,...,1,Divulgada no PNCP,Compras.gov.br,NaN,10959,220,134,86,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...
